# AutoExpert: Intelligent Vehicle Diagnosis & Mechanic Expert System
## Academic Demonstration, Symbolic Inference Verification & Evaluation Notebook

**Subject:** Expert Systems / Knowledge-Based Systems  
**Domain:** Automotive & Motorcycle Mechanical Diagnosis  
**Architecture:** Pure Symbolic Expert System (Forward Chaining, Backward Chaining, 3VL Working Memory, 5-Tier Conflict Resolution)  
**Core Engine:** `autoexpert_engine` (Shared Python Package)  

---

### Abstract & System Overview
AutoExpert is a deterministic, rule-based expert system designed to simulate the diagnostic reasoning of an experienced automotive and motorcycle technician. Unlike machine-learning black boxes or probabilistic language models, AutoExpert operates strictly through formal knowledge engineering:
1. **Structured Knowledge Base:** 55+ validated IF-THEN rules covering Engine, Electrical, Transmission, Brakes, Cooling & Lubrication, Suspension & Tyres, Fuel Systems, and Maintenance.
2. **Three-Valued Logic (3VL) Working Memory:** Tracks facts across `TRUE`, `FALSE`, and `UNKNOWN`, strictly preserving fact provenance and never falsifying unknown information.
3. **Multi-Pass Forward Chaining:** Deductive, data-driven reasoning iterating until a fixed point is reached, demonstrating multi-hop rule activation.
4. **Recursive Backward Chaining:** Goal-driven hypothesis testing with cycle detection (`visited_goals`) and automated missing-fact question generation.
5. **5-Tier Conflict Resolution:** Deterministic arbitration across Refraction, Safety Criticality, Specificity, Explicit Priority, and Rule ID.
6. **Safety-First Escalation:** Immediate halt upon detecting critical hazards (brake failure, low oil pressure, fuel leaks, extreme overheating).

## Section 1: Colab Environment Setup & Package Installation
In Google Colab, execute the cell below to clone the repository (if running in the cloud) or install `autoexpert_engine` in editable mode. Both the FastAPI web backend and this notebook import the exact same package with zero code duplication.

In [ ]:
# Google Colab Setup Cell
import os
import sys
from pathlib import Path

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab environment...")
    if not os.path.exists("autoexpert"):
        os.system("git clone https://github.com/your-username/autoexpert.git")
        os.chdir("autoexpert")
    os.system("pip install -e .")
else:
    # Local repository execution
    repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(repo_root))
    print(f"Local environment: sys.path augmented with {repo_root}")

# Verify shared engine package import
import autoexpert_engine
from autoexpert_engine import (
    KnowledgeBase,
    WorkingMemory,
    ForwardChainingEngine,
    BackwardChainingEngine,
    ConflictResolver,
    DiagnosticEngine,
    ExplanationEngine,
    FactProvenance,
    ProofStatus,
)
print(f"SUCCESS: autoexpert_engine v{autoexpert_engine.__version__} successfully imported!")

## Section 2: Knowledge Base Loading & JSON Schema Validation
Here we load the rules from `knowledge_base/motorcycle_rules.json`, `knowledge_base/car_rules.json`, and `knowledge_base/maintenance_rules.json`, validating every rule against `rule_schema.json` via `jsonschema`.

In [ ]:
# Initialize KnowledgeBase and inspect statistics
kb = KnowledgeBase()
stats = kb.stats()

print("=" * 60)
print("           AUTOEXPERT KNOWLEDGE BASE OVERVIEW")
print("=" * 60)
print(f"Total Validated Rules : {stats['total_rules']}")
print(f"Safety Critical Rules : {stats['safety_critical_rules']}")
print("Distribution by Diagnostic Category:")
for cat, count in stats["by_category"].items():
    print(f"  - {cat:<22}: {count} rule(s)")
print("=" * 60)

# Inspect a representative multi-hop rule
sample_rule = kb.get_rule("MOTO_ELEC_001")
print(f"\nSample Rule Inspection [{sample_rule.rule_id}]:")
print(f"  Name         : {sample_rule.name}")
print(f"  Category     : {sample_rule.category.value}")
print(f"  Conditions   : {[f'{c.fact} {c.operator} {c.value}' for c in sample_rule.conditions]}")
print(f"  Derived Facts: {[f'{df.fact}={df.value}' for df in sample_rule.derived_facts]}")
print(f"  Conclusion   : {sample_rule.conclusion.title if sample_rule.conclusion else None}")
print(f"  Safety Crit  : {sample_rule.safety_critical} | Priority: {sample_rule.priority}")

## Section 3: Three-Valued Logic (3VL) Working Memory Demonstration
Traditional Boolean systems erroneously assume that an unknown fact is false (the Closed-World Assumption). In medical and automotive diagnosis, this is catastrophic. AutoExpert implements strict **Three-Valued Logic (3VL)**:
- `TRUE`: Confirmed by user observation, symptom, or diagnostic measurement.
- `FALSE`: Confirmed absent or refuted.
- `UNKNOWN`: Not yet assessed or unknown to the user. `UNKNOWN` is never coerced to `FALSE`.

In [ ]:
# Demonstrate 3VL Working Memory behavior
wm = WorkingMemory()

# 1. Unasserted fact is UNKNOWN
print("Is 'starter_clicks' known?  :", wm.is_known('starter_clicks'))
print("Is 'starter_clicks' unknown?:", wm.is_unknown('starter_clicks'))
print("Is 'starter_clicks' false?  :", wm.is_false('starter_clicks'))

# 2. Assert facts with provenance
wm.assert_fact("starter_clicks", True, provenance=FactProvenance.OBSERVATION)
wm.assert_fact("battery_voltage", 11.8, provenance=FactProvenance.MEASUREMENT)

# 3. Inspect provenance metadata
fact_obj = wm.get_fact("battery_voltage")
print(f"\nFact: {fact_obj.key} = {fact_obj.value} {type(fact_obj.value)}")
print(f"Provenance: {fact_obj.provenance.value} | Pass: {fact_obj.pass_number}")

# 4. Mutation history audit log
print("\nWorking Memory Mutation History:")
for entry in wm.get_mutation_history():
    print(f"  [{entry['type']}] {entry['key']} -> {entry.get('value')} (provenance: {entry['provenance']})")

## Section 4: Multi-Pass Forward Chaining & 5-Tier Conflict Resolution
Forward chaining begins from initial facts and repeatedly applies eligible rules until a fixed point is reached. When multiple rules are eligible in a pass, the 5-Tier Conflict Resolution engine deterministically selects the winner:
1. **Tier 1 — Refraction:** Rules that have fired in this session cannot refire on the same fact state.
2. **Tier 2 — Safety Criticality:** Rules tagged `safety_critical: True` take absolute precedence.
3. **Tier 3 — Specificity:** Rules with more preconditions (higher specificity) take precedence.
4. **Tier 4 — Explicit Priority:** Highest numerical rule priority wins.
5. **Tier 5 — Lexicographical Rule ID:** Alphabetical ordering ensures 100% deterministic tiebreaking.

In [ ]:
# Demonstrate Multi-Pass Forward Chaining on Multi-Hop Scenario (SV650 Stator Breakdown)
engine = DiagnosticEngine()

initial_facts = {
    "engine_starts": False,
    "starter_clicks": True,
    "headlights_dim": True,
    "battery_recharged_recently": True,
    "stator_ac_voltage_low": True,
}
vehicle = {"vehicle_type": "motorcycle", "cooling_type": "liquid_cooled", "fuel_type": "petrol"}

result = engine.run_forward_chaining(initial_facts, vehicle_context=vehicle)

print(f"Inference Runtime: {result.runtime_ms:.2f} ms | Passes: {result.total_passes} | Fixed Point: {result.fixed_point_reached}")
print(f"Triggered Rules Sequence: {' -> '.join(result.triggered_rules)}")
print("\nStep-by-Step Multi-Hop Execution Trace:")
for step in result.execution_steps:
    print(f"\nPass {step.pass_number}: Fired Rule '{step.rule_id}' ({step.rule_name})")
    print(f"  Premise Satisfied: {', '.join([c['fact'] for c in step.satisfied_conditions])}")
    if step.derived_facts_added:
        print(f"  --> Asserted Intermediate Derived Facts: {[d['fact'] for d in step.derived_facts_added]}")
    if step.conclusion_reached:
        print(f"  ==> Concluded Fault: {step.conclusion_reached['title']} [{step.conclusion_reached['fault_id']}]")

## Section 5: Goal-Driven Recursive Backward Chaining & Cycle Detection
Backward chaining starts with a hypothetical fault (e.g., `FAULT_WEAK_BATTERY`) and recursively works backwards to verify if evidence supports it. If intermediate facts are missing, the engine identifies them as **missing observable facts** and outputs targeted interview questions.
To prevent infinite recursion in cyclic knowledge bases (e.g., Rule A -> Goal B and Rule B -> Goal A), the engine maintains `visited_goals` on the recursion call stack.

In [ ]:
# Backward Chaining Demonstration
bc_engine = BackwardChainingEngine(kb.rules)

# Case A: Fully Proven Goal
wm_proven = WorkingMemory({"engine_starts": False, "starter_clicks": True, "headlights_dim": True})
proof_a = bc_engine.prove_goal("FAULT_WEAK_BATTERY", wm_proven, vehicle_context=vehicle)
print("Goal: FAULT_WEAK_BATTERY with complete evidence:")
print(f"  Proof Status: {proof_a.status.value}")

# Case B: Inconclusive with Missing Facts
wm_incomplete = WorkingMemory({"starter_clicks": True})
proof_b = bc_engine.prove_goal("FAULT_WEAK_BATTERY", wm_incomplete, vehicle_context=vehicle)
print("\nGoal: FAULT_WEAK_BATTERY with incomplete evidence:")
print(f"  Proof Status : {proof_b.status.value}")
print(f"  Missing Facts: {proof_b.missing_facts}")
print("  Generated Targeted Questions:")
for q in proof_b.suggested_questions:
    print(f"    - [{q['fact_key']}] {q['question_text']}")

## Section 6: Explanation Facility ("Why" and "How" Introspection)
A hallmark of true expert systems is explanation transparency. AutoExpert explains:
- **Why** a rule fired (satisfied premises in working memory).
- **How** each conclusion was established across multi-hop inference.
- **Conflict Resolution Rationale** for every pass.

In [ ]:
# Explanation Engine Demo
explanation = ExplanationEngine.explain_forward_chaining(result, WorkingMemory(initial_facts))
print(f"Safety Escalation Active: {explanation['safety_escalation']}")
print("\nDetailed 'How' Conclusions Explanations:")
for how in explanation["how_explanations"]:
    print(f"\n--- {how['title']} ({how['fault_id']}) ---")
    print(how['explanation'])
    print(f"Severity: {how['severity']} | Recommended Action: {how['recommended_action']}")

## Section 7: Execution of All 10+ Academic Diagnostic Scenarios
We execute all 10 academic scenarios from `knowledge_base/demo_scenarios.json` against `DiagnosticEngine`, comparing expected outcomes with actual inference results.

In [ ]:
import json
import pandas as pd

scenario_path = Path("knowledge_base/demo_scenarios.json")
if not scenario_path.exists():
    scenario_path = Path.cwd().parent / "knowledge_base" / "demo_scenarios.json"

with open(scenario_path, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

evaluation_records = []

print(f"Running {len(scenarios)} Defined Academic Diagnostic Scenarios...\n")

for sc in scenarios:
    res = engine.run_forward_chaining(sc["initial_facts"], vehicle_context=sc["vehicle_profile"])
    actual_faults = [c["fault_id"] for c in res.conclusions]
    expected_faults = sc["expected_conclusions"]
    
    # Check match
    conclusions_matched = all(ef in actual_faults for ef in expected_faults)
    actual_safety = any(c.get("safety_critical") for c in res.conclusions)
    safety_matched = (actual_safety == sc["expected_safety_escalation"])
    all_passed = conclusions_matched and safety_matched
    
    evaluation_records.append({
        "scenario_id": sc["scenario_id"],
        "title": sc["title"],
        "category": sc["category"],
        "vehicle_type": sc["vehicle_profile"]["vehicle_type"],
        "expected_conclusions": ", ".join(expected_faults),
        "actual_conclusions": ", ".join(actual_faults),
        "rules_triggered": len(res.triggered_rules),
        "passes": res.total_passes,
        "runtime_ms": res.runtime_ms,
        "verdict": "MATCH (PASS)" if all_passed else "MISMATCH (FAIL)"
    })

eval_df = pd.DataFrame(evaluation_records)
print(eval_df[["scenario_id", "rules_triggered", "passes", "runtime_ms", "verdict"]].to_string(index=False))

## Section 8: Evaluation Metrics & Academic Discussion
> [!IMPORTANT]
> **Academic Scope Clarification:** The evaluation metrics computed below measure deterministic verification and consistency against the **predefined academic test scenario suite (10 scenarios)**. In accordance with knowledge engineering standards, these metrics reflect **test-set accuracy, not real-world statistical accuracy across all automotive faults in existence**.

In [ ]:
# Compute test-set evaluation metrics
total_scenarios = len(eval_df)
passed_scenarios = sum(eval_df["verdict"] == "MATCH (PASS)")
test_set_accuracy = (passed_scenarios / total_scenarios) * 100.0
avg_runtime = eval_df["runtime_ms"].mean()
avg_rules_fired = eval_df["rules_triggered"].mean()

print("=" * 65)
print("            AUTOEXPERT TEST-SET EVALUATION METRICS")
print("=" * 65)
print(f"Total Test Scenarios Evaluated  : {total_scenarios}")
print(f"Scenarios Passing Verification : {passed_scenarios} / {total_scenarios}")
print(f"Test-Set Rule Match Accuracy   : {test_set_accuracy:.1f}%")
print(f"Mean Inference Execution Time  : {avg_runtime:.3f} ms")
print(f"Mean Rules Fired Per Scenario  : {avg_rules_fired:.1f}")
print("=" * 65)
print("Note: Test-set accuracy reflects adherence to expert ground truth for designed scenarios.")

## Section 9: Matplotlib Visualizations
Visualizing rule activation frequencies, scenario execution runtime distributions, and diagnostic categories across the test suite.

In [ ]:
import matplotlib.pyplot as plt

# Chart 1: Rule Count by Diagnostic Category
plt.style.use('dark_background') if 'dark_background' in plt.style.available else None
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cats = list(stats['by_category'].keys())
counts = list(stats['by_category'].values())

bars = ax1.barh(cats, counts, color='#3B82F6', edgecolor='#1D4ED8')
ax1.set_title('Knowledge Base Rule Distribution by Category', fontsize=12, fontweight='bold', pad=10)
ax1.set_xlabel('Number of Rules')
ax1.grid(axis='x', linestyle='--', alpha=0.3)
for bar in bars:
    width = bar.get_width()
    ax1.text(width + 0.2, bar.get_y() + bar.get_height()/2, f'{int(width)}', va='center', fontsize=9)

# Chart 2: Scenario Inference Runtime (ms)
scenario_names = [s.replace('SCENARIO_', '') for s in eval_df['scenario_id']]
ax2.bar(range(len(scenario_names)), eval_df['runtime_ms'], color='#10B981', edgecolor='#047857')
ax2.set_title('Inference Runtime per Scenario (ms)', fontsize=12, fontweight='bold', pad=10)
ax2.set_xlabel('Scenario Index')
ax2.set_ylabel('Execution Time (ms)')
ax2.set_xticks(range(len(scenario_names)))
ax2.set_xticklabels(range(1, len(scenario_names) + 1))
ax2.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/autoexpert_evaluation_charts.png', dpi=150)
if IN_COLAB or 'ipykernel' in sys.modules:
    plt.show()
else:
    plt.close(fig)
print("Evaluation charts generated and saved to notebooks/autoexpert_evaluation_charts.png")

## Section 10: Conclusion, Limitations & Future Work

### Academic Findings
1. **Deterministic Traceability:** AutoExpert derives every mechanical conclusion with 100% mathematical auditability through rule execution traces and working memory mutation logs.
2. **3VL Reliability:** Three-valued logic eliminates premature diagnosis by preserving `UNKNOWN` states until conclusive evidence is provided.
3. **Multi-Hop Inference:** Separating intermediate derived conditions (e.g., `power_supply_insufficient` or `coolant_loss_detected`) from final conclusions enables clean multi-pass reasoning that mirrors diagnostic troubleshooting.
4. **Safety Prioritization:** Deterministic 5-tier conflict resolution ensures safety-critical conditions supersede routine maintenance suggestions in high-risk scenarios.

### Known Limitations & Future Work
- **Closed Ground Truth:** The knowledge base currently covers 55 validated rules. Unmodeled vehicle faults (e.g., CAN-bus multiplexing faults) remain inconclusive.
- **Certainty Factors (CF):** Future work could incorporate MYCIN-style certainty factor propagation to express nuanced degrees of confidence in ambiguous sensor readings.
- **OBD-II Telemetry Streaming:** Integration with live ELM327 Bluetooth/USB scanners to assert diagnostic trouble codes (DTCs) and freeze-frame sensor values directly into working memory.